In [4]:
import torch, gc
gc.collect(); torch.cuda.empty_cache()

# install bitsandbytes for 4-bit quantization
import subprocess
subprocess.run("pip install bitsandbytes --quiet".split())

from transformers import Qwen2AudioForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

quant = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

print("Loading Qwen2-Audio in 4-bit (~5GB)...")
pal_proc  = AutoProcessor.from_pretrained("Qwen/Qwen2-Audio-7B-Instruct")
pal_model = Qwen2AudioForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-Audio-7B-Instruct",
    quantization_config=quant,
    device_map={"": 0},
)
print("Qwen loaded in 4-bit ✓")

free, _ = torch.cuda.mem_get_info()
print(f"GPU free after load: {free/1e9:.1f}GB")

Loading Qwen2-Audio in 4-bit (~5GB)...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/876 [00:00<?, ?it/s]

Qwen loaded in 4-bit ✓
GPU free after load: 8.7GB


In [6]:
import json, numpy as np, pandas as pd, librosa, torch
import torch.nn as nn

CLASSES = ["Scream","Shout","Crying","Explosion","Gunshot","Glass","Siren","Alarm"]
SR, N_MELS, TIME_FRAMES = 16000, 128, 128
HOP = 10.0/32
DEVICE = "cuda"

if "IDX" not in dir():
    IDX = json.load(open("audioset_index.json"))

# SED model
class CRNN_v3(nn.Module):
    def __init__(self, n, sed=True):
        super().__init__()
        self.sed = sed
        def blk(i,o,pool=(2,2)):
            return nn.Sequential(
                nn.Conv2d(i,o,3,padding=1), nn.BatchNorm2d(o), nn.ReLU(),
                nn.Conv2d(o,o,3,padding=1), nn.BatchNorm2d(o), nn.ReLU(),
                nn.MaxPool2d(pool), nn.Dropout2d(0.1))
        self.cnn  = nn.Sequential(blk(1,32,(2,2)), blk(32,64,(2,2)), blk(64,128,(2,1)))
        self.lstm = nn.LSTM(128*16, 128, batch_first=True, bidirectional=True, num_layers=2, dropout=0.2)
        self.drop = nn.Dropout(0.4)
        self.fc   = nn.Linear(256, n)
    def forward(self, x):
        x = self.cnn(x); b,c,f,t = x.size()
        x = x.permute(0,3,1,2).contiguous().view(b,t,c*f)
        x,_ = self.lstm(x); x = self.drop(x)
        return self.fc(x) if self.sed else self.fc(x.mean(1))

sed_model = CRNN_v3(8, sed=True).to(DEVICE)
sed_model.load_state_dict(torch.load("best_audioset_sed_v3.pth"))
sed_model.eval()

BEST_THR = {'Scream':0.82,'Shout':0.81,'Crying':0.92,'Explosion':0.75,
            'Gunshot':0.61,'Glass':0.82,'Siren':0.80,'Alarm':0.62}

def run_sed(ytid):
    path = IDX.get(ytid, ytid)
    y,_ = librosa.load(path, sr=SR, duration=10.0)
    mel = librosa.power_to_db(librosa.feature.melspectrogram(y=y, sr=SR, n_mels=N_MELS))
    mel = (mel-mel.mean())/(mel.std()+1e-6)
    mel = np.pad(mel,((0,0),(0,TIME_FRAMES-mel.shape[1]))) if mel.shape[1]<TIME_FRAMES else mel[:,:TIME_FRAMES]
    x = torch.tensor(mel[np.newaxis,np.newaxis], dtype=torch.float32).to(DEVICE)
    with torch.no_grad():
        with torch.amp.autocast("cuda"):
            return torch.sigmoid(sed_model(x))[0].float().cpu().numpy()

def get_events(probs, thr, min_dur=0.3):
    ev = []
    for j,cls in enumerate(CLASSES):
        act, inev, st, sc = probs[:,j]>thr[cls], False, 0, []
        for t,a in enumerate(act):
            if a and not inev: st, inev, sc = t*HOP, True, [probs[t,j]]
            elif a and inev: sc.append(probs[t,j])
            elif not a and inev:
                if t*HOP-st>=min_dur: ev.append((cls,st,t*HOP,float(max(sc))))
                inev=False
        if inev: ev.append((cls,st,10.0,float(max(sc))))
    return sorted(ev, key=lambda e:e[1])

test_df = pd.read_csv("audioset_v2_test.csv")
test_df["ytid"] = test_df["ytid"].str.strip()

print("SED model + test_df loaded ✓")
print("Test clips:", len(test_df))

SED model + test_df loaded ✓
Test clips: 1249


/tmp/ipykernel_4024040/1364815748.py:33: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  sed_model.load_state_dict(torch.load("best_audioset_sed_v3.pth"))


In [8]:
def pal_warning_llm(audio_path, sed_events):
    y, sr = librosa.load(audio_path, sr=16000)
    if sed_events:
        det = ", ".join(f"{c} at {s:.0f}-{e:.0f}s" for c,s,e,_ in sed_events)
        hint = f"An automatic detector flagged: {det}. "
    else:
        hint = ""
    prompt = (hint +
        "Listen to this audio and write a brief, factual content warning "
        "for a media viewer. Describe what distressing sounds are present "
        "and roughly when. One or two sentences. Do not speculate beyond "
        "what is audible.")
    conv = [{"role":"user","content":[
        {"type":"audio","audio_url":audio_path},
        {"type":"text","text":prompt}]}]
    text = pal_proc.apply_chat_template(conv, add_generation_prompt=True, tokenize=False)
    inputs = pal_proc(text=text, audios=[y], sampling_rate=sr, return_tensors="pt")
    inputs = {k: v.to(pal_model.device) for k, v in inputs.items()}
    with torch.no_grad():
        out = pal_model.generate(**inputs, max_new_tokens=120)
    reply = pal_proc.batch_decode(out[:, inputs["input_ids"].shape[1]:],
                                  skip_special_tokens=True)[0]
    return reply.strip()

print("pal_warning_llm ready.")

pal_warning_llm ready.


In [9]:
import json
saved = []

for target in ["Siren", "Gunshot", "Crying"]:
    row = test_df[test_df[target]==1].iloc[0]
    path = IDX[row["ytid"]]
    events = get_events(run_sed(row["ytid"]), BEST_THR, 0.3)

    warning = pal_warning_llm(path, events)

    result = {
        "clip": row["ytid"],
        "true_label": target,
        "sed_events": [(c, round(s,1), round(e,1)) for c,s,e,_ in events],
        "qwen_warning": warning
    }
    saved.append(result)
    print(f"\n=== {target} ({row['ytid']}) ===")
    print("SED:", result["sed_events"])
    print("Qwen PAL:", warning)

with open("pal_qwen_outputs.json", "w") as f:
    json.dump(saved, f, indent=2)
print("\n✓ Saved to pal_qwen_outputs.json — evidence secured")

[transformers] Keyword argument `audios` is not a valid argument for this processor and will be ignored.



=== Siren (vfUgQTKgKDI) ===
SED: [('Alarm', 0.9, 5.6), ('Siren', 1.2, 2.5), ('Siren', 3.4, 6.2)]
Qwen PAL: The audio contains alarming sounds of sirens from 1.02 to 2.00 and from 3.04 to 6.00, followed by a continuous alarm sound from 1.00 to 6.00.

=== Gunshot (Dj9gyAoqmQ0) ===
SED: [('Gunshot', 1.2, 2.2)]
Qwen PAL: Gunshot is heard at approximately 1.24 to 1.89 seconds into the audio.

=== Crying (jr2kxASSRBY) ===
SED: [('Scream', 3.1, 3.4)]
Qwen PAL: "Attention viewers, there are distressing sounds of a scream occurring from 3.24 to 3.86 seconds in the audio."

✓ Saved to pal_qwen_outputs.json — evidence secured
